# Air Quality Classification Using Decision Stump

## Academic Information

**Student Name:** Abdullah  
**Roll No:** 1688  
**Semester:** 4th Semester  
**University:** Govt. Abdul Ali Khan College, Charsadda  
**Course Project:** Machine Learning  
**Course Instructor / Supervisor:** Mr. Sakin Jan

---

## Project Overview
This project uses a **Decision Stump (a Decision Tree with depth = 1)** to classify air quality as **Good** or **Poor** using environmental sensor data.

### Features Used
- Temperature (TEMP)
- Humidity (Hum)
- Carbon Dioxide (CO2)
- Carbon Monoxide (CO)

### Learning Type
**Supervised Machine Learning**

The model learns from labeled examples where the correct air-quality class is already known. After training, it can predict the class of new samples.


**Goal:** Predict if air quality is **Good (0)** or **Poor (1)**  
**Features:** TEMP, Hum, CO2, CO  
**Model:** Decision Tree with max_depth=1 (called a Decision Stump)

---


## Step 1: Import Libraries
We bring in all the tools we need.

###  Explanation – 
**Library Import**

**Purpose:** Import all required Python libraries.

**What This Code Does:**
- Loads Pandas for data handling.
- Loads Matplotlib and Seaborn for graphs.
- Loads Scikit-learn tools for machine learning.

**Why Important?**
Every machine learning project depends on libraries for data processing, visualization, model training, and evaluation.

**Expected Output:** No visible output. Libraries become available for use.

**Machine Learning Significance:** These libraries form the foundation of the complete workflow.


In [14]:
import pandas as pd                        # for loading and working with data
import matplotlib.pyplot as plt            # for drawing charts
import seaborn as sns                      # for heatmap
import warnings
warnings.filterwarnings('ignore')          # hide unnecessary warnings

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

print('All libraries loaded!')

All libraries loaded!


## Step 2: Load the Dataset
We read the CSV file. Make sure it is in the **same folder** as this notebook.

###  Explanation 
**Dataset Loading**

**Purpose:** Load the Air Quality dataset into memory.

**What This Code Does:**
- Reads the CSV file.
- Stores the data in a DataFrame.
- Removes unnecessary columns if present.
- Displays sample records.

**Why Important?**
Machine learning cannot begin until the data is loaded and inspected.

**Expected Output:** Dataset table and basic information.

**Machine Learning Significance:** High-quality data preparation leads to better model performance.


In [15]:
df = pd.read_csv('AirQuality_data.csv')    # load the dataset

if 'Unnamed: 0' in df.columns:            # remove extra index column if present
    df.drop(columns=['Unnamed: 0'], inplace=True)

print('Shape:', df.shape)                  # rows x columns
print()
print('Class counts:')
print(df['class'].value_counts())          # how many Good vs Poor
df.head()                                  # show first 5 rows

Shape: (2989, 5)

Class counts:
class
0    2189
1     800
Name: count, dtype: int64


,TEMP,Hum,CO2,CO,class
0,16.0,57,370,2.1,0
1,16.0,57,360,2.1,0
2,16.0,57,380,2.1,0
3,15.0,58,390,2.0,0
4,15.0,58,380,2.0,0


## Step 3: Chart 1 — Pie Chart (Class Distribution)
Shows how many samples are Good Air vs Poor Air as percentages.

###  Explanation 
**Pie Chart**

**Purpose:** Visualize class distribution.

**What This Graph Shows:**
The percentage of Good Air and Poor Air samples.

**Key Observations:**
Helps identify whether the dataset is balanced or imbalanced.

**Importance for Analysis:**
Balanced datasets generally produce more reliable models.


In [16]:
counts = df['class'].value_counts()        # count Good(0) and Poor(1)

plt.figure(figsize=(5, 5))
plt.pie(
    counts,                                # data values
    labels=['Good Air (0)', 'Poor Air (1)'],
    colors=['#2ecc71', '#e74c3c'],         # green = good, red = poor
    autopct='%1.1f%%',                     # show percentage on each slice
    startangle=90
)
plt.title('Class Distribution')
plt.show()

## Step 4: Chart 2 — Bar Chart (Class Count)
Same info as pie chart but as bars — easier to see exact numbers.

###  Explanation 
**Bar Chart**


**Purpose:** Display exact class counts.

**What This Graph Shows:**
The number of records belonging to each air-quality category.

**Why Important?**
Provides a clearer comparison than percentages alone.

**Machine Learning Significance:**
Class distribution influences model training and evaluation.


In [17]:
plt.figure(figsize=(5, 4))
plt.bar(
    ['Good Air (0)', 'Poor Air (1)'],      # x-axis labels
    counts.values,                         # heights of bars
    color=['#2ecc71', '#e74c3c']           # green and red
)
plt.title('Class Count')
plt.ylabel('Number of Samples')
plt.show()

## Step 5: Chart 3 — Correlation Heatmap
Shows how much each feature is related to others.  
Value close to **1** = strong positive link | close to **-1** = strong negative link | **0** = no link.

###  Explanation 
**Correlation Heatmap**

**Purpose:** Understand relationships between variables.

**What This Graph Shows:**
Correlation values between features and target variables.

**Key Observations:**
- Positive values indicate direct relationships.
- Negative values indicate inverse relationships.

**Importance for Analysis:**
Helps identify useful features for prediction.


In [18]:
plt.figure(figsize=(6, 5))
sns.heatmap(
    df.corr(),                             # calculate correlation between all columns
    annot=True,                            # show numbers inside boxes
    fmt='.2f',                             # 2 decimal places
    cmap='RdYlGn'                          # red=negative, yellow=zero, green=positive
)
plt.title('Correlation Heatmap')
plt.show()

## Step 6: Prepare Data + Train/Test Split
X = input features, y = what we want to predict.  
We split **80% for training** and **20% for testing**.

###  Explanation 
**Data Preparation**

**Purpose:** Prepare data for machine learning.

**What This Code Does:**
- Selects input features.
- Defines the target class.
- Splits data into training and testing sets.

**Why Important?**
Separating training and testing data prevents biased evaluation.

**Expected Output:** X_train, X_test, y_train, and y_test datasets.

**Machine Learning Significance:** This is a standard supervised-learning practice.


In [19]:
X = df[['TEMP', 'Hum', 'CO2', 'CO']]      # input features
y = df['class']                            # target (what to predict)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,                         # 20% goes to test
    random_state=42,                       # fixed seed so results are same every run
    stratify=y                             # keep same Good/Poor ratio in both splits
)

print(f'Train samples: {X_train.shape[0]}')
print(f'Test  samples: {X_test.shape[0]}')

Train samples: 2391
Test  samples: 598


## Step 7: Train the Decision Stump
max_depth=1 means the tree makes only **ONE decision** using **ONE feature**.  
Think of it like: *if CO2 is high → Poor Air, else → Good Air*.

###  Explanation 
*** Decision Stump Training**


**Purpose:** Train the classification model.

**What This Code Does:**
Creates a Decision Tree with maximum depth = 1.

**Why Important?**
A Decision Stump makes only one decision rule, making it simple and interpretable.

**Expected Output:** Trained machine learning model.

**Machine Learning Significance:** The model learns patterns from historical air-quality data.


In [20]:
model = DecisionTreeClassifier(
    max_depth=1,                           # only 1 split = decision stump
    criterion='gini',                      # gini measures how mixed the classes are
    random_state=42
)
model.fit(X_train, y_train)               # train the model on training data

feature_names = ['TEMP', 'Hum', 'CO2', 'CO']
best_feature  = feature_names[model.tree_.feature[0]]   # which feature was chosen
threshold     = model.tree_.threshold[0]                 # the split value
print(f'Decision rule: if {best_feature} <= {threshold:.4f} then Good Air, else Poor Air')

Decision rule: if CO2 <= 155.0000 then Good Air, else Poor Air


## Step 8: Evaluate the Model
Check accuracy and detailed report on the test data (data the model has never seen).

### Professional Explanation – Model Evaluation

**Purpose:** Measure model performance.

**What This Code Does:**
- Generates predictions.
- Calculates accuracy.
- Produces a classification report.

**Why Important?**
Evaluation determines whether the model is reliable.

**Expected Output:** Accuracy, Precision, Recall, and F1-Score.

**Machine Learning Significance:** Helps assess prediction quality.


In [21]:
y_pred = model.predict(X_test)            # make predictions on test data

acc = accuracy_score(y_test, y_pred)      # compare predictions vs real labels
print(f'Accuracy: {acc*100:.2f}%')
print()
# precision = of all predicted Poor, how many were really Poor?
# recall    = of all actual Poor, how many did we catch?
# f1-score  = balance between precision and recall
print(classification_report(y_test, y_pred, target_names=['Good Air', 'Poor Air']))

Accuracy: 97.66%

              precision    recall  f1-score   support

    Good Air       0.98      0.98      0.98       438
    Poor Air       0.96      0.96      0.96       160

    accuracy                           0.98       598
   macro avg       0.97      0.97      0.97       598
weighted avg       0.98      0.98      0.98       598



## Step 9: Chart 4 — Confusion Matrix
A 2x2 table showing correct vs wrong predictions.  
- **Top-left** = correctly predicted Good Air  
- **Bottom-right** = correctly predicted Poor Air  
- **Other boxes** = mistakes

### Professional Explanation – Confusion Matrix

**Purpose:** Analyze prediction results in detail.

**What This Graph Shows:**
- True Positives
- True Negatives
- False Positives
- False Negatives

**Why Important?**
Shows exactly where the model succeeds and fails.

**Machine Learning Significance:** Useful for improving model performance.


In [22]:
cm = confusion_matrix(y_test, y_pred)     # calculate the confusion matrix

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Good Air', 'Poor Air']
)
disp.plot(cmap='Blues')                   # darker blue = more samples
plt.title('Confusion Matrix')
plt.show()

## Step 10: Chart 5 — Feature Importance
Since it is a Decision Stump, only **1 feature** is used.  
That feature gets importance = 1.0, all others = 0.

### explanation –
**Feature Importance**

**Purpose:** Identify the most influential feature.

**What This Code Does:**
Calculates feature importance scores from the trained model.

**Why Important?**
Shows which sensor measurement contributes most to prediction.

**Expected Output:** Importance values and graph.

**Machine Learning Significance:** Helps understand model decision-making.


In [23]:
importances = model.feature_importances_  # importance score for each feature

plt.figure(figsize=(6, 4))
plt.bar(
    feature_names,
    importances,
    color=['#e74c3c' if v > 0 else '#bdc3c7' for v in importances]  # red = used feature
)
plt.title('Feature Importance')
plt.ylabel('Importance Score')
plt.show()

## Step 11: Visualize the Decision Tree
Shows the actual tree diagram with the single split rule.

 **Explanation**

  **Decision** Tree Visualization

**Purpose:** Display the learned decision rule.

**What This Graph Shows:**
The feature selected by the Decision Stump and its threshold value.

**Why Important?**
Provides transparency and interpretability.

**Machine Learning Significance:** Makes model decisions easy to explain during viva presentations.


In [24]:
plt.figure(figsize=(10, 4))
plot_tree(
    model,
    feature_names=feature_names,
    class_names=['Good Air', 'Poor Air'],
    filled=True,                           # color nodes by class
    rounded=True,                          # rounded box corners
    fontsize=12
)
plt.title('Decision Stump Tree (max_depth=1)')
plt.show()

## Step 12: Predict on a New Sample
Enter any values and see what the model predicts.  
You can change the numbers below to test different scenarios!

 New Sample Prediction

**Purpose:** Test the trained model on unseen data.

**What This Code Does:**
Creates a new sample and predicts its air-quality category.

**Why Important?**
Demonstrates real-world usage of the trained model.

**Expected Output:** Good Air or Poor Air prediction.

**Machine Learning Significance:** Shows practical deployment capability.


In [25]:
# change these values to test!
new_sample = pd.DataFrame({
    'TEMP': [22.5],
    'Hum':  [60],
    'CO2':  [500],
    'CO':   [2.3]
})

prediction  = model.predict(new_sample)[0]           # 0 = Good, 1 = Poor
probability = model.predict_proba(new_sample)[0]     # confidence scores

print('Input:', new_sample.to_dict('records')[0])
print(f'Good Air chance : {probability[0]*100:.1f}%')
print(f'Poor Air chance : {probability[1]*100:.1f}%')
print('Result:', 'GOOD AIR' if prediction == 0 else 'POOR AIR - Warning!')

Input: {'TEMP': 22.5, 'Hum': 60, 'CO2': 500, 'CO': 2.3}
Good Air chance : 98.6%
Poor Air chance : 1.4%
Result: GOOD AIR



##  Project Summary



###  What We Did — Step by Step

| Step | Action |
|------|--------|
| 1 | Imported all required Python libraries |
| 2 | Loaded the Air Quality dataset from CSV |
| 3 | Visualized class distribution (Pie Chart) |
| 4 | Visualized class counts (Bar Chart) |
| 5 | Analyzed feature relationships (Correlation Heatmap) |
| 6 | Prepared features (X) and target (y), split 80% train / 20% test |
| 7 | Trained a **Decision Stump** (Decision Tree with max_depth=1) |
| 8 | Evaluated model using Accuracy, Precision, Recall, F1-Score |
| 9 | Visualized predictions using Confusion Matrix |
| 10 | Plotted Feature Importance |
| 11 | Drew the Decision Tree structure |
| 12 | Predicted air quality for a new custom sample |



### Key Concepts Learned

| Concept | Meaning |
|---------|---------|
| **Decision Stump** | Simplest classifier — makes ONE decision using ONE feature and ONE threshold |
| **Supervised Learning** | Model learns from labeled data (inputs + correct answers provided) |
| **Gini Impurity** | Measures how mixed classes are at a node (0 = pure, 0.5 = most mixed) |
| **Train/Test Split** | Keeps test data unseen during training for honest evaluation |
| **Accuracy** | Percentage of total correct predictions |
| **Confusion Matrix** | Table showing True/False Positives and Negatives |
| **Feature Importance** | How much each feature contributed to the model's decision |



###  Key Results

- **Model Used:** Decision Stump (DecisionTreeClassifier, max_depth=1)
- **Best Feature Selected:** The single feature the model found most useful for splitting
- **Decision Rule:** `IF best_feature <= threshold → Good Air (0), ELSE → Poor Air (1)`



> **Tip:** A Decision Stump is fast and highly interpretable but limited in power.  
> Try increasing `max_depth=3` or `max_depth=5` and compare the accuracy to see how a deeper tree performs!